In [1]:
import random
import torch

In [2]:
random.seed(67)
torch.manual_seed(67)

Let's start with a batch of xs with batch_size = 2, seq_len = 3, d_emb = 4.  So that's a (2, 3, 4) tensor:

In [3]:
batch_size = 2
seq_len = 3
d_emb = 4

In [4]:
xs = torch.rand((batch_size, seq_len, d_emb))

In [5]:
xs.shape

torch.Size([2, 3, 4])

Now, let's say we have seven experts.  We get weights with a linear layer mapping from d_emb to num_experts.

In [6]:
num_experts = 7

In [7]:
router = torch.nn.Linear(d_emb, num_experts, bias=False)

In [8]:
routing_logits = router(xs)

In [9]:
routing_logits.shape

torch.Size([2, 3, 7])

As expected, it's (batch_size, seq_len, num_experts)

In [10]:
routing_logits

tensor([[[ 0.1627, -0.3312,  0.7548,  0.3239, -0.4476,  0.4251, -0.3319],
         [ 0.0934, -0.4510,  0.9410,  0.3338, -0.4869,  0.5465, -0.2595],
         [ 0.2307, -0.2847,  0.7082,  0.1746, -0.3453,  0.3372, -0.4413]],

        [[ 0.0398, -0.2237,  0.6263,  0.2902, -0.3324,  0.3782, -0.1889],
         [ 0.0599, -0.2611,  0.2460,  0.0296, -0.1585,  0.1370, -0.0216],
         [ 0.0427, -0.2502,  0.5480,  0.1718, -0.2585,  0.3135, -0.1475]]],
       grad_fn=<UnsafeViewBackward0>)

Let's have five active -- very artificial, but it means that all of our axis sizes are different which should make things easier to compare.

In [11]:
num_active_experts = 5

In [12]:
top_k_values, top_k_indices = torch.topk(
    routing_logits, 
    k=num_active_experts,
    dim=-1, sorted=True
)

In [13]:
top_k_values.shape

torch.Size([2, 3, 5])

In [14]:
top_k_values

tensor([[[ 0.7548,  0.4251,  0.3239,  0.1627, -0.3312],
         [ 0.9410,  0.5465,  0.3338,  0.0934, -0.2595],
         [ 0.7082,  0.3372,  0.2307,  0.1746, -0.2847]],

        [[ 0.6263,  0.3782,  0.2902,  0.0398, -0.1889],
         [ 0.2460,  0.1370,  0.0599,  0.0296, -0.0216],
         [ 0.5480,  0.3135,  0.1718,  0.0427, -0.1475]]],
       grad_fn=<TopkBackward0>)

They're sorted descending, so we can get the lowest value for each one easily.  However, we want something with the same number of axes as the routing_logits so that it can later be broadcast across routing_logits, so instest of having -1 at the end, we have -1:

In [15]:
lowest_top_k_values = top_k_values[:, :, -1:]

In [16]:
lowest_top_k_values.shape

torch.Size([2, 3, 1])

In [17]:
lowest_top_k_values

tensor([[[-0.3312],
         [-0.2595],
         [-0.2847]],

        [[-0.1889],
         [-0.0216],
         [-0.1475]]], grad_fn=<SliceBackward0>)

So now we can get a mask showing which of the routing_logits are not in the top 5

In [18]:
not_top_k_mask = routing_logits < lowest_top_k_values

In [19]:
not_top_k_mask

tensor([[[False, False, False, False,  True, False,  True],
         [False,  True, False, False,  True, False, False],
         [False, False, False, False,  True, False,  True]],

        [[False,  True, False, False,  True, False, False],
         [False,  True, False, False,  True, False, False],
         [False,  True, False, False,  True, False, False]]])

If you compare that with the routing_logits up above, you can see that it has indeed got `True`s in the positions for the lowest two.

We're going to want to run the logits through softmax to get per-expert weights.  But we want the non-top-k experts -- the two per token that we have got `True`s for in `not_top_k_mask` to be set to zero weight.  So we do the normal trick of setting them to `-torch.inf`.

In [20]:
masked_routing_logits = routing_logits.masked_fill(not_top_k_mask, -torch.inf)

In [21]:
masked_routing_logits.shape

torch.Size([2, 3, 7])

In [22]:
masked_routing_logits

tensor([[[ 0.1627, -0.3312,  0.7548,  0.3239,    -inf,  0.4251,    -inf],
         [ 0.0934,    -inf,  0.9410,  0.3338,    -inf,  0.5465, -0.2595],
         [ 0.2307, -0.2847,  0.7082,  0.1746,    -inf,  0.3372,    -inf]],

        [[ 0.0398,    -inf,  0.6263,  0.2902,    -inf,  0.3782, -0.1889],
         [ 0.0599,    -inf,  0.2460,  0.0296,    -inf,  0.1370, -0.0216],
         [ 0.0427,    -inf,  0.5480,  0.1718,    -inf,  0.3135, -0.1475]]],
       grad_fn=<MaskedFillBackward0>)

That looks solid, so we can softmax it.

In [23]:
expert_weights = torch.softmax(masked_routing_logits, dim=-1)

In [24]:
expert_weights.shape

torch.Size([2, 3, 7])

In [25]:
expert_weights

tensor([[[0.1697, 0.1036, 0.3068, 0.1994, 0.0000, 0.2206, 0.0000],
         [0.1453, 0.0000, 0.3392, 0.1848, 0.0000, 0.2286, 0.1021],
         [0.1899, 0.1134, 0.3060, 0.1795, 0.0000, 0.2112, 0.0000]],

        [[0.1592, 0.0000, 0.2862, 0.2045, 0.0000, 0.2233, 0.1267],
         [0.1932, 0.0000, 0.2327, 0.1874, 0.0000, 0.2087, 0.1781],
         [0.1685, 0.0000, 0.2794, 0.1918, 0.0000, 0.2210, 0.1394]]],
       grad_fn=<SoftmaxBackward0>)

OK, let's see how we'd work with expert 1.  It's the second column in each of those batch items above.  It has two active experts.

In [26]:
expert_ix = 1

Firstly, let's see which items should go through it?

In [27]:
this_expert_mask = expert_weights[:, :, expert_ix] > 0

In [28]:
this_expert_mask.shape

torch.Size([2, 3])

So that's (batch_size, seq_len) and shows which tokens should go through expert 1.

In [29]:
this_expert_mask

tensor([[ True, False,  True],
        [False, False, False]])

So that matches the right sequences

In [30]:
xs

tensor([[[0.7380, 0.3646, 0.8673, 0.5464],
         [0.5885, 0.9785, 0.8875, 0.9903],
         [0.3595, 0.3245, 0.8871, 0.9991]],

        [[0.5664, 0.5706, 0.7217, 0.2157],
         [0.0415, 0.2903, 0.0052, 0.8127],
         [0.2556, 0.6512, 0.5186, 0.6332]]])

In [31]:
xs[this_expert_mask]

tensor([[0.7380, 0.3646, 0.8673, 0.5464],
        [0.3595, 0.3245, 0.8871, 0.9991]])

...and that is indeed the batch that we want to run through -- the first and the third tokens from the first sequence in the batch.  Let's fake up the results to that.  In our case, our results are the same shape as the inputs, so let's make it all ones.

In [32]:
this_expert_results = torch.ones_like(xs[this_expert_mask])

In [33]:
this_expert_results.shape

torch.Size([2, 4])

So that's shaped (number_of_tokens_for_this_expert, d_emb)

In [34]:
this_expert_results

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]])

Now, we want to multiply those by the appropriate weights (the non-zero values in the two second columns of the `expert_weights` tensor above).  And then we'll want to aggregate them with other results from other experts.  A neat way of doing that is to start of with all-zero results in the same shape as our inputs (again, we're relying on inputs and outputs having the same shape) and then to add these, weighted, into the right slots in there.  Let's do that step by step.

In [35]:
all_outputs = torch.zeros_like(xs)

Let's get the weights we want to multiply them by, in the same shape as our results.

In [36]:
this_expert_weights = expert_weights[this_expert_mask, expert_ix].unsqueeze(1)

In [37]:
this_expert_weights.shape

torch.Size([2, 1])

In [38]:
this_expert_weights

tensor([[0.1036],
        [0.1134]], grad_fn=<UnsqueezeBackward0>)

In [39]:
this_expert_results * this_expert_weights

tensor([[0.1036, 0.1036, 0.1036, 0.1036],
        [0.1134, 0.1134, 0.1134, 0.1134]], grad_fn=<MulBackward0>)

So, we have our output results, each one weighted by the weight that the specified token had for the specified expert.  Now we can add them in to the outputs

In [40]:
all_outputs[this_expert_mask] += this_expert_results * this_expert_weights

In [41]:
all_outputs

tensor([[[0.1036, 0.1036, 0.1036, 0.1036],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.1134, 0.1134, 0.1134, 0.1134]],

        [[0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000]]], grad_fn=<IndexPutBackward0>)